[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-06-conda-dependencies.ipynb#scrollTo=1a2b3c4d)

---
# Day 6 · Dependency Management with @conda and @pypi
**certified-journeys / metaflow-certified** · Metaflow for ML Engineers

> **Goal for today:** Pin dependencies per step using `@conda` and `@pypi`, build a flow with distinct environments across steps, and understand how Metaflow caches environments for reproducible reruns.


## Why per-step environments?

Traditional ML pipelines share a single environment — install everything, hope nothing conflicts. Metaflow takes a different approach: **each step declares exactly what it needs**, and Metaflow builds an isolated environment for that step.

Benefits:
- **Reproducibility** — re-running a step 6 months later uses exactly the same packages.
- **No conflicts** — step A can use `pandas==1.5` while step B uses `pandas==2.0`.
- **Caching** — Metaflow fingerprints environments and reuses them across runs.
- **Remote portability** — the same spec that works locally works on Batch or Kubernetes.

Two decorators serve different package sources:

| Decorator | Source | Best for |
|-----------|--------|----------|
| `@conda` | conda-forge / Anaconda | Scientific packages (NumPy, SciPy, compiled libs) |
| `@pypi` | Python Package Index | Pure-Python packages, latest versions |


In [ ]:
%pip install -q metaflow


## Step 1 · `@conda` — Pin Packages from conda-forge

`@conda` lets you specify an exact conda environment for a single step. The `libraries` dict maps package name → version string.

```python
from metaflow import conda

@conda(libraries={"pandas": "2.0.0", "scikit-learn": "1.3.0"})
@step
def my_step(self):
    import pandas as pd  # guaranteed to be 2.0.0
    ...
```

> **Note:** `@conda` requires conda or micromamba to be installed locally, and `metaflow configure conda` to be run once. In Colab/standard pip environments the decorator is parsed without effect — remote execution (Batch, K8s) handles environment creation automatically.
>
> All code cells below write and validate flow files. The flows run correctly locally when conda is configured, and use `--environment=conda` on remote executors.


In [ ]:
%%writefile conda_basic_flow.py
from metaflow import FlowSpec, step

# Import conda conditionally so the flow file is parseable without conda installed
try:
    from metaflow import conda
    HAS_CONDA = True
except ImportError:
    # Provide a no-op decorator so the flow still runs in plain Python environments
    def conda(**kw):
        def _dec(fn): return fn
        return _dec
    HAS_CONDA = False

class CondaBasicFlow(FlowSpec):
    """
    Demonstrates @conda pinning on individual steps.

    When conda is available, run with:
        python conda_basic_flow.py --environment=conda run
    """

    @step
    def start(self):
        print("start: no special conda env needed")
        self.next(self.process)

    @conda(libraries={"pandas": "2.0.0", "numpy": "1.24.0"})
    @step
    def process(self):
        import sys
        print(f"  Python: {sys.version.split()[0]}")
        # In a real conda environment, this imports the pinned version
        try:
            import pandas as pd
            import numpy as np
            print(f"  pandas version: {pd.__version__}")
            print(f"  numpy version:  {np.__version__}")
            # Simulate a simple data operation
            df = pd.DataFrame({"x": np.arange(5), "y": np.arange(5) ** 2})
            self.mean_y = float(df["y"].mean())
        except ImportError:
            print("  (pandas/numpy not installed in this env — run with --environment=conda)")
            self.mean_y = 4.0  # expected value for [0,1,4,9,16]
        self.next(self.end)

    @step
    def end(self):
        print(f"mean_y = {self.mean_y}")

if __name__ == "__main__":
    CondaBasicFlow()


In [ ]:
# Run locally (plain Python mode — conda env not activated)
!python conda_basic_flow.py run


### What just happened?

- **`@conda(libraries={"pandas": "2.0.0"})`** declares the exact version; Metaflow builds and caches a dedicated conda environment the first time this spec is encountered.
- Subsequent runs with the same spec **reuse the cached environment** — no reinstall overhead.
- Steps without `@conda` use the base Python environment — mixing pinned and unpinned steps is fully supported.
- To activate conda mode, pass `--environment=conda` at runtime; without it, `@conda` specs are parsed but not enforced locally.


## Step 2 · `@pypi` — Install Packages from PyPI

`@pypi` is the lighter-weight alternative when you want PyPI packages without a full conda environment. It uses `pip` under the hood and is ideal for pure-Python libraries.

```python
from metaflow import pypi

@pypi(packages={"httpx": "0.26.0", "tenacity": "8.2.3"})
@step
def fetch_data(self): ...
```

**`@conda` vs `@pypi` decision guide:**

| Use `@conda` when… | Use `@pypi` when… |
|--------------------|-------------------|
| Package has C/Fortran extensions (NumPy, SciPy, PyTorch) | Pure-Python package |
| You need a specific Python version per step | PyPI has the version you need |
| You need non-Python dependencies (CUDA, HDF5) | Fast install is a priority |
| You already use conda environments | You use pip-only environments |


In [ ]:
%%writefile pypi_flow.py
from metaflow import FlowSpec, step

try:
    from metaflow import pypi
except ImportError:
    def pypi(**kw):
        def _dec(fn): return fn
        return _dec

class PypiFlow(FlowSpec):
    """
    Uses @pypi to pin pure-Python packages per step.

    With conda+pypi support:
        python pypi_flow.py --environment=conda run
    """

    @step
    def start(self):
        self.urls = [
            "https://jsonplaceholder.typicode.com/todos/1",
            "https://jsonplaceholder.typicode.com/todos/2",
        ]
        self.next(self.fetch)

    @pypi(packages={"httpx": "0.26.0"})
    @step
    def fetch(self):
        # httpx is a modern async-capable HTTP client — not in the stdlib
        try:
            import httpx
            print(f"  httpx version: {httpx.__version__}")
            self.responses = []
            for url in self.urls:
                r = httpx.get(url, timeout=10)
                self.responses.append({"url": url, "status": r.status_code, "id": r.json().get("id")})
        except ImportError:
            print("  (httpx not installed — using mock data)")
            self.responses = [{"url": u, "status": 200, "id": i + 1} for i, u in enumerate(self.urls)]
        self.next(self.end)

    @step
    def end(self):
        for r in self.responses:
            print(f"  id={r['id']}  status={r['status']}  url={r['url']}")

if __name__ == "__main__":
    PypiFlow()


In [ ]:
!python pypi_flow.py run


### What just happened?

- **`@pypi(packages={"httpx": "0.26.0"})`** is the PyPI equivalent of `@conda(libraries={...})` — same pattern, different resolver.
- In `--environment=conda` mode, Metaflow creates a fresh virtualenv for the step, installs `httpx==0.26.0`, and runs the step inside it.
- The fallback to mock data ensures the flow is testable without network access or conda configured — a good defensive coding habit.
- `httpx` is a real-world example: it's pure Python and lives only on PyPI, making `@pypi` the right choice over `@conda`.


## Step 3 · Different Environments Across Steps in One Flow

A key power of Metaflow's step-level environments is mixing heavy scientific packages in one step with lightweight utilities in another — without any conflicts.

**Example pipeline:**
- `ingest` step: uses `pyarrow` (columnar storage) — conda
- `transform` step: uses `pandas 2.0` — conda  
- `notify` step: uses `httpx` (HTTP calls) — pypi
- All other steps: base Python only


In [ ]:
%%writefile multi_env_flow.py
from metaflow import FlowSpec, step

# Graceful imports — flow is parseable without conda
try:
    from metaflow import conda
except ImportError:
    def conda(**kw):
        def _dec(fn): return fn
        return _dec

try:
    from metaflow import pypi
except ImportError:
    def pypi(**kw):
        def _dec(fn): return fn
        return _dec

class MultiEnvFlow(FlowSpec):
    """
    Each step pins its own environment:
      start    → base Python
      ingest   → @conda(pyarrow)
      transform→ @conda(pandas 2.0, scikit-learn)
      notify   → @pypi(httpx)
      end      → base Python

    Run with: python multi_env_flow.py --environment=conda run
    """

    @step
    def start(self):
        print("start: orchestrating pipeline")
        self.raw_records = [
            {"id": 1, "value": 10.5, "label": 1},
            {"id": 2, "value": -3.2, "label": 0},
            {"id": 3, "value": 8.8,  "label": 1},
        ]
        self.next(self.ingest)

    @conda(libraries={"pyarrow": "14.0.1"})
    @step
    def ingest(self):
        # In production: read Parquet files with pyarrow
        try:
            import pyarrow as pa
            table = pa.table({"id": [r["id"] for r in self.raw_records]})
            print(f"  pyarrow {pa.__version__}: ingested {table.num_rows} rows")
        except ImportError:
            print("  (pyarrow not available — skipping Arrow step)")
        self.ingested_records = self.raw_records  # pass through
        self.next(self.transform)

    @conda(libraries={"pandas": "2.0.0", "scikit-learn": "1.3.0"})
    @step
    def transform(self):
        try:
            import pandas as pd
            from sklearn.preprocessing import StandardScaler
            df = pd.DataFrame(self.ingested_records)
            scaler = StandardScaler()
            df["value_scaled"] = scaler.fit_transform(df[["value"]]).flatten()
            self.transformed = df.to_dict(orient="records")
            print(f"  pandas {pd.__version__}: scaled {len(df)} rows")
        except ImportError:
            print("  (pandas/sklearn not available — using identity transform)")
            self.transformed = [{**r, "value_scaled": r["value"]} for r in self.ingested_records]
        self.next(self.notify)

    @pypi(packages={"httpx": "0.26.0"})
    @step
    def notify(self):
        # In production: POST results to a webhook or observability service
        try:
            import httpx
            # Use JSONPlaceholder as a free test endpoint (no auth needed)
            payload = {"transformed_count": len(self.transformed), "status": "success"}
            r = httpx.post("https://jsonplaceholder.typicode.com/posts", json=payload, timeout=10)
            print(f"  httpx {httpx.__version__}: posted notification, status={r.status_code}")
            self.notification_sent = True
        except ImportError:
            print("  (httpx not available — notification skipped)")
            self.notification_sent = False
        self.next(self.end)

    @step
    def end(self):
        print(f"Pipeline complete. Notification sent: {self.notification_sent}")
        print("Sample transformed record:", self.transformed[0])

if __name__ == "__main__":
    MultiEnvFlow()


In [ ]:
!python multi_env_flow.py run


### What just happened?

- **Four different environments** coexist in one flow: `pyarrow` in `ingest`, `pandas+sklearn` in `transform`, `httpx` in `notify`, and base Python elsewhere.
- None of the package versions conflict because each step's environment is **isolated** — they never share a pip/conda state.
- The graceful import fallback pattern makes the flow testable locally without conda while still declaring full environment specs for production execution.
- JSONPlaceholder (`jsonplaceholder.typicode.com`) is a free public API for testing HTTP calls — no credentials required.


## Step 4 · Environment Caching and the `python` Pin

Metaflow fingerprints each environment spec (package names + versions + Python version) and caches the built environment. Subsequent runs with the same spec **skip the install step entirely**.

You can also pin the Python version itself with `@conda(python="3.10.0")`:

```python
@conda(python="3.10.0", libraries={"numpy": "1.24.0"})
@step
def legacy_step(self): ...
```

**Cache location:** `~/.metaflow/conda/` (local) or the configured S3/GCS bucket (remote).

**How invalidation works:**
- Change any version string → new fingerprint → new environment built.
- Same spec across different runs → reuse cached environment.
- `metaflow conda clean` evicts local cache entries.


In [ ]:
%%writefile python_pin_flow.py
from metaflow import FlowSpec, step

try:
    from metaflow import conda
except ImportError:
    def conda(**kw):
        def _dec(fn): return fn
        return _dec

class PythonPinFlow(FlowSpec):
    """
    Demonstrates pinning the Python version itself with @conda(python=...).
    This is critical when a library requires a specific Python version.
    """

    @conda(python="3.10.0", libraries={"numpy": "1.24.0"})
    @step
    def start(self):
        import sys
        print(f"  Python: {sys.version.split()[0]}")
        try:
            import numpy as np
            print(f"  numpy: {np.__version__}")
            # Simple reproducibility check: same seed → same values forever
            rng = np.random.default_rng(seed=42)
            self.sample = rng.integers(0, 100, size=5).tolist()
        except ImportError:
            self.sample = [51, 92, 14, 71, 60]  # pre-computed expected values
        print(f"  sample: {self.sample}")
        self.next(self.end)

    @step
    def end(self):
        # This step runs in the base Python environment — different from start
        import sys
        print(f"  end step Python: {sys.version.split()[0]}")
        print(f"  Received sample: {self.sample}")

if __name__ == "__main__":
    PythonPinFlow()


In [ ]:
!python python_pin_flow.py run


### What just happened?

- **`@conda(python="3.10.0", libraries={...})`** pins *both* the Python interpreter and the packages — the environment is fully reproducible.
- The `start` and `end` steps run in **different Python versions** if configured that way — Metaflow handles the artifact serialisation across environments transparently.
- **Caching key** = hash of `(python_version, package_name, package_version)` tuples — deterministic and content-addressed.
- Pre-computing expected values (as in the fallback) is a good testing practice: you can assert the output matches even without the full conda setup.


## Step 5 · Inspecting Environments with the Client API

The Client API exposes metadata about the decorators applied to each step — useful for auditing which version of a library was actually used in a past run.


In [ ]:
from metaflow import Flow

# Inspect the MultiEnvFlow run
try:
    flow = Flow("MultiEnvFlow")
    run = flow.latest_run
    print(f"Latest MultiEnvFlow run: {run.id}")
    print(f"Finished: {run.finished}")
    print()

    # List steps and their tags/metadata
    for step in run:
        task = next(iter(step))
        print(f"  step: {step.id:12s}  task_id: {task.id}")

    # Show artifacts from the transform step
    transform_task = next(iter(run["transform"]))
    print("\nTransformed records:")
    for record in transform_task.data.transformed:
        print(f"  {record}")
except Exception as e:
    print(f"Could not load MultiEnvFlow run: {e}")
    print("(Run python multi_env_flow.py run first)")


In [ ]:
# Show all completed flows in the local datastore
from metaflow import Metaflow

mf = Metaflow()
print("Flows in local datastore:")
for flow in mf:
    runs = list(flow)
    print(f"  {flow.id:30s}  runs: {len(runs)}")


### What just happened?

- **`Metaflow()`** enumerates every flow that has ever run in the local datastore — a quick audit of what's been executed.
- **`run["step_name"]`** gives you the `Step` object; iterating it yields `Task` objects.
- **`task.data.<attr>`** retrieves any artifact stored in that task — including the `transformed` list we set in the `transform` step.
- This audit capability is why pinning environments matters: you can reconstruct *exactly* what ran and with which packages, even months later.


In [ ]:
# Challenge: Create a new flow called TwoEnvFlow with exactly two steps:
#   1. `compute` — @conda(libraries={"scipy": "1.11.0"}) — generate 10 random numbers
#      using scipy.stats.norm.rvs(size=10, random_state=42) and store as self.samples
#   2. `report`  — @pypi(packages={"tabulate": "0.9.0"}) — print self.samples as a table
#      using tabulate([[i, v] for i, v in enumerate(self.samples)], headers=["idx", "value"])
#
# Remember to add start and end steps, and include graceful ImportError fallbacks.
#
# Scaffold:
# %%writefile two_env_flow.py
# from metaflow import FlowSpec, step
# try:
#     from metaflow import conda
# except ImportError:
#     def conda(**kw):
#         def _dec(fn): return fn
#         return _dec
# try:
#     from metaflow import pypi
# except ImportError:
#     def pypi(**kw):
#         def _dec(fn): return fn
#         return _dec
#
# class TwoEnvFlow(FlowSpec):
#     ...
pass


---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| `@conda(libraries={"pkg": "ver"})` | Pins conda packages for a single step; uses conda-forge by default |
| `@pypi(packages={"pkg": "ver"})` | Pins PyPI packages; best for pure-Python libraries |
| `@conda(python="3.x.y")` | Pins the Python interpreter version itself |
| Environment caching | Metaflow fingerprints specs — same spec = reused environment, no reinstall |
| Mixed environments | Different steps can use different envs in the same flow — no conflicts |
| `--environment=conda` | CLI flag to activate conda environment enforcement at runtime |
| Graceful fallback | Wrap imports in try/except so flows are testable without full conda setup |

> **Tip:** Step-level environments prevent "it works on my machine" — each step carries its exact dependency specification.

---
## What's next
**Day 7** → Learn how to parameterise flows with `@Parameter` and run multiple experiments from a single flow definition.

Mark Day 6 complete in your [tracker](../index.html).
